# 🔄 End-to-End ML Pipeline for Customer Churn Prediction
## DevelopersHub Corporation — AI/ML Engineering Internship
### Task 2: Scikit-learn Pipeline API · GridSearchCV · Joblib Export

**Objective:** Build a reusable, production-ready ML pipeline to predict customer churn using the **Telco Customer Churn Dataset**.

| Item | Detail |
|------|--------|
| Dataset | Telco Customer Churn (IBM / Kaggle) |
| Models | Logistic Regression · Random Forest |
| Tuning | GridSearchCV with Stratified K-Fold |
| Export | joblib (production-ready `.pkl` files) |
| Skills | Pipeline API · Preprocessing · Hyperparameter Tuning · Model Deployment |

---
> ⚠️ **Google Colab Users:** Runtime → Change Runtime Type → T4 GPU (optional but faster)

In [ ]:
# ============================================================
# CELL 1 ▸ Install Required Libraries
# ============================================================

!pip install scikit-learn pandas numpy matplotlib seaborn joblib -q

## Step 1 · Import Libraries

In [ ]:
# ============================================================
# CELL 2 ▸ Import All Required Libraries
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings("ignore")

# Scikit-learn — Pipeline & Preprocessing
from sklearn.pipeline        import Pipeline
from sklearn.compose         import ColumnTransformer
from sklearn.preprocessing   import StandardScaler, OneHotEncoder
from sklearn.impute          import SimpleImputer

# Models
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier

# Tuning & Evaluation
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     StratifiedKFold, cross_val_score)
from sklearn.metrics         import (accuracy_score, f1_score, roc_auc_score,
                                     classification_report, confusion_matrix,
                                     ConfusionMatrixDisplay, roc_curve)

print("✅ All libraries imported successfully!")

## Step 2 · Load the Dataset
We load the **Telco Customer Churn** dataset directly from GitHub (IBM repository).  
It contains **7,043 customers** and **20 features** including demographics, services, and billing info.

In [ ]:
# ============================================================
# CELL 3 ▸ Load Telco Churn Dataset
# ============================================================

URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

print("📥 Loading Telco Churn dataset...")
df = pd.read_csv(URL)

print(f"\n✅ Dataset loaded!")
print(f"   Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n📋 Columns:\n{df.columns.tolist()}")
print(f"\n🎯 Target (Churn):")
print(df['Churn'].value_counts())
print(f"\n   Churn rate: {(df['Churn']=='Yes').mean()*100:.1f}%")

## Step 3 · Exploratory Data Analysis (EDA)

In [ ]:
# ============================================================
# CELL 4 ▸ Exploratory Data Analysis
# ============================================================

# --- Basic info ---
print("=" * 55)
print("📊 DATASET OVERVIEW")
print("=" * 55)
print(df.info())
print(f"\n🔍 Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\n📈 Numeric stats:")
print(df[['tenure','MonthlyCharges','TotalCharges']].describe().round(2))

# --- Visualizations ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Telco Churn — Exploratory Data Analysis", fontsize=15, fontweight="bold", y=1.01)

colors = {"Yes": "#E74C3C", "No": "#2ECC71"}

# 1. Churn distribution
churn_counts = df['Churn'].value_counts()
axes[0,0].pie(churn_counts, labels=['No Churn','Churn'],
              colors=['#2ECC71','#E74C3C'], autopct='%1.1f%%',
              startangle=90, textprops={'fontsize':12})
axes[0,0].set_title("Churn Distribution", fontsize=12, fontweight="bold")

# 2. Tenure by churn
for churn_val, color in colors.items():
    axes[0,1].hist(df[df['Churn']==churn_val]['tenure'],
                   bins=30, alpha=0.7, label=churn_val, color=color, edgecolor='black', linewidth=0.3)
axes[0,1].set_title("Tenure Distribution by Churn", fontsize=12, fontweight="bold")
axes[0,1].set_xlabel("Tenure (months)"); axes[0,1].legend(title="Churn")
axes[0,1].grid(True, alpha=0.3)

# 3. Monthly charges by churn
df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[0,2],
           patch_artist=True,
           boxprops=dict(facecolor='#4A90D9', alpha=0.6))
axes[0,2].set_title("Monthly Charges by Churn", fontsize=12, fontweight="bold")
axes[0,2].set_xlabel("Churn"); axes[0,2].set_ylabel("Monthly Charges ($)")
plt.sca(axes[0,2]); plt.title("Monthly Charges by Churn")

# 4. Contract type vs churn
contract_churn = df.groupby(['Contract','Churn']).size().unstack(fill_value=0)
contract_churn.plot(kind='bar', ax=axes[1,0], color=['#2ECC71','#E74C3C'],
                    edgecolor='black', linewidth=0.5)
axes[1,0].set_title("Contract Type vs Churn", fontsize=12, fontweight="bold")
axes[1,0].set_xlabel("Contract"); axes[1,0].set_ylabel("Count")
axes[1,0].tick_params(axis='x', rotation=15); axes[1,0].legend(title="Churn")
axes[1,0].grid(True, alpha=0.3, axis='y')

# 5. Internet service vs churn
internet_churn = df.groupby(['InternetService','Churn']).size().unstack(fill_value=0)
internet_churn.plot(kind='bar', ax=axes[1,1], color=['#2ECC71','#E74C3C'],
                    edgecolor='black', linewidth=0.5)
axes[1,1].set_title("Internet Service vs Churn", fontsize=12, fontweight="bold")
axes[1,1].set_xlabel("Service"); axes[1,1].set_ylabel("Count")
axes[1,1].tick_params(axis='x', rotation=15); axes[1,1].legend(title="Churn")
axes[1,1].grid(True, alpha=0.3, axis='y')

# 6. Correlation heatmap (numeric)
df_num = df[['tenure','MonthlyCharges','TotalCharges','SeniorCitizen']].copy()
df_num['Churn'] = (df['Churn']=='Yes').astype(int)
sns.heatmap(df_num.corr(), ax=axes[1,2], annot=True, fmt=".2f",
            cmap="RdYlGn", center=0, linewidths=0.5, square=True)
axes[1,2].set_title("Correlation Heatmap", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("eda_plots.png", dpi=130, bbox_inches="tight")
plt.show()
print("✅ EDA plots saved!")

## Step 4 · Data Cleaning & Feature Engineering
Key issues to fix:
- `TotalCharges` is stored as string — contains whitespace for new customers (tenure=0)
- Drop `customerID` (non-informative identifier)
- Encode target `Churn` as binary integer

In [ ]:
# ============================================================
# CELL 5 ▸ Data Cleaning
# ============================================================

# Drop customer ID — not a predictive feature
df = df.drop(columns=['customerID'])

# Fix TotalCharges: whitespace strings → NaN → imputed later in pipeline
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f"TotalCharges NaN (new customers): {df['TotalCharges'].isna().sum()}")
# These NaNs will be handled by SimpleImputer(strategy='median') in our pipeline

# Encode target: 'Yes' → 1, 'No' → 0
y = (df['Churn'] == 'Yes').astype(int)
X = df.drop(columns=['Churn'])

print(f"\n✅ Cleaning done!")
print(f"   Features : {X.shape[1]} columns")
print(f"   Samples  : {len(X):,}")
print(f"\n🎯 Target distribution:")
print(f"   No Churn (0) : {(y==0).sum():,} ({(y==0).mean()*100:.1f}%)")
print(f"   Churn    (1) : {(y==1).sum():,} ({(y==1).mean()*100:.1f}%)")

# Identify numeric and categorical columns
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"\n📊 Numeric features  ({len(num_cols)}): {num_cols}")
print(f"📝 Categorical features ({len(cat_cols)}): {cat_cols}")

## Step 5 · Build Preprocessing + Model Pipelines
This is the core of the task — a fully encapsulated `Pipeline` that:
1. **Imputes** missing values
2. **Scales** numeric features
3. **Encodes** categorical features
4. **Trains** the classifier

All steps are chained together so the same object handles raw data in production.

In [ ]:
# ============================================================
# CELL 6 ▸ Build Scikit-learn Pipelines
# ============================================================

# --- Numeric sub-pipeline: impute median → standard scale ---
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # Handles TotalCharges NaNs
    ('scaler',  StandardScaler()),                  # Zero mean, unit variance
])

# --- Categorical sub-pipeline: impute mode → one-hot encode ---
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),   # Fill any missing cats
    ('encoder', OneHotEncoder(handle_unknown='ignore',      # Ignore unseen categories
                               sparse_output=False)),        # Return dense array
])

# --- Column transformer: applies each sub-pipeline to its columns ---
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols),
], remainder='drop')

# --- Full pipelines: preprocessor + classifier ---
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(max_iter=1000, random_state=42)),
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(random_state=42, n_jobs=-1)),
])

print("✅ Pipelines built!")
print("\n🔧 Logistic Regression Pipeline:")
print(lr_pipeline)
print("\n🔧 Random Forest Pipeline:")
print(rf_pipeline)

## Step 6 · Train / Test Split

In [ ]:
# ============================================================
# CELL 7 ▸ Train / Test Split (Stratified)
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 80% train, 20% test
    random_state=42,
    stratify=y           # Preserve class ratio in both splits
)

print(f"✅ Split complete!")
print(f"   Training set : {len(X_train):,} samples  ({y_train.mean()*100:.1f}% churn)")
print(f"   Test set     : {len(X_test):,}  samples  ({y_test.mean()*100:.1f}% churn)")

## Step 7 · Baseline Training (Before Tuning)

In [ ]:
# ============================================================
# CELL 8 ▸ Baseline Training (Default Hyperparameters)
# ============================================================

print("🏋️  Training baseline models...")

lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

print("\n" + "=" * 55)
print("📊 BASELINE RESULTS (Default Hyperparameters)")
print("=" * 55)

for name, pipe in [("Logistic Regression", lr_pipeline), ("Random Forest", rf_pipeline)]:
    preds = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    acc   = accuracy_score(y_test, preds)
    f1    = f1_score(y_test, preds)
    auc   = roc_auc_score(y_test, proba)
    print(f"\n  {name}:")
    print(f"    Accuracy : {acc:.4f} ({acc*100:.2f}%)")
    print(f"    F1-Score : {f1:.4f}")
    print(f"    ROC-AUC  : {auc:.4f}")

print("\n⚙️  Now running GridSearchCV for hyperparameter tuning...")

## Step 8 · Hyperparameter Tuning with GridSearchCV
We search over key hyperparameters using **5-Fold Stratified Cross-Validation**.
Scoring metric: **F1-Score** (better for imbalanced classes than accuracy).

In [ ]:
# ============================================================
# CELL 9 ▸ GridSearchCV Hyperparameter Tuning
# ============================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- Logistic Regression search space ---
lr_param_grid = {
    'classifier__C':      [0.01, 0.1, 1.0, 10.0],   # Regularization strength
    'classifier__solver': ['lbfgs', 'liblinear'],     # Optimization algorithm
}

# --- Random Forest search space ---
rf_param_grid = {
    'classifier__n_estimators':    [100, 200],        # Number of trees
    'classifier__max_depth':       [None, 10, 20],    # Tree depth limit
    'classifier__min_samples_split': [2, 5],          # Min samples to split a node
}

# ── Run GridSearch: Logistic Regression ──────────────────────
print("🔍 GridSearchCV — Logistic Regression  (8 combinations × 5 folds = 40 fits)...")
lr_gs = GridSearchCV(
    lr_pipeline,
    lr_param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)
lr_gs.fit(X_train, y_train)

print(f"\n✅ Best LR params : {lr_gs.best_params_}")
print(f"   Best CV F1     : {lr_gs.best_score_:.4f}")

# ── Run GridSearch: Random Forest ───────────────────────────
print("\n🔍 GridSearchCV — Random Forest  (12 combinations × 5 folds = 60 fits)...")
rf_gs = GridSearchCV(
    rf_pipeline,
    rf_param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)
rf_gs.fit(X_train, y_train)

print(f"\n✅ Best RF params : {rf_gs.best_params_}")
print(f"   Best CV F1     : {rf_gs.best_score_:.4f}")

## Step 9 · Final Evaluation on Test Set

In [ ]:
# ============================================================
# CELL 10 ▸ Final Evaluation (Tuned Models)
# ============================================================

best_models = {
    "Logistic Regression": lr_gs.best_estimator_,
    "Random Forest":       rf_gs.best_estimator_,
}

print("=" * 60)
print("📊 FINAL TEST SET RESULTS (After Tuning)")
print("=" * 60)

results = {}
for name, model in best_models.items():
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'accuracy': accuracy_score(y_test, preds),
        'f1':       f1_score(y_test, preds),
        'auc':      roc_auc_score(y_test, proba),
        'preds':    preds,
        'proba':    proba,
    }
    print(f"\n{'─'*40}")
    print(f"  {name}")
    print(f"{'─'*40}")
    print(f"  Accuracy : {results[name]['accuracy']:.4f}")
    print(f"  F1-Score : {results[name]['f1']:.4f}")
    print(f"  ROC-AUC  : {results[name]['auc']:.4f}")
    print(f"\n{classification_report(y_test, preds, target_names=['No Churn','Churn'])}")

# Pick the best overall model
best_name = max(results, key=lambda k: results[k]['auc'])
print(f"\n🏆 Best Model by AUC: {best_name}  ({results[best_name]['auc']:.4f})")

## Step 10 · Confusion Matrices

In [ ]:
# ============================================================
# CELL 11 ▸ Confusion Matrix Plots
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Confusion Matrices — Tuned Models", fontsize=14, fontweight="bold")

for ax, (name, model) in zip(axes, best_models.items()):
    cm = confusion_matrix(y_test, model.predict(X_test))
    ConfusionMatrixDisplay(cm, display_labels=['No Churn', 'Churn']).plot(
        ax=ax, cmap="Blues", colorbar=False
    )
    acc = results[name]['accuracy']
    ax.set_title(f"{name}\nAccuracy: {acc*100:.2f}%", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Confusion matrices saved!")

## Step 11 · ROC Curves

In [ ]:
# ============================================================
# CELL 12 ▸ ROC Curve Comparison
# ============================================================

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#4A90D9', '#E74C3C']

for (name, model), color in zip(best_models.items(), colors):
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f"{name}  (AUC = {auc:.4f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier (AUC = 0.50)')
ax.fill_between([0,1],[0,1], alpha=0.05, color='gray')
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — Model Comparison", fontsize=14, fontweight="bold")
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ ROC curves saved!")

## Step 12 · Feature Importance (Random Forest)

In [ ]:
# ============================================================
# CELL 13 ▸ Feature Importance — Random Forest
# ============================================================

rf_best    = best_models["Random Forest"]
ohe_feats  = (rf_best
              .named_steps['preprocessor']
              .named_transformers_['cat']['encoder']
              .get_feature_names_out(cat_cols))
all_feats   = np.concatenate([num_cols, ohe_feats])
importances = rf_best.named_steps['classifier'].feature_importances_

# Top 15 features
top_idx = np.argsort(importances)[-15:][::-1]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(all_feats[top_idx][::-1], importances[top_idx][::-1],
               color='#4A90D9', edgecolor='black', linewidth=0.5)
ax.set_title("Top 15 Feature Importances — Random Forest", fontsize=13, fontweight="bold")
ax.set_xlabel("Importance Score", fontsize=11)
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for bar, val in zip(bars, importances[top_idx][::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n🔑 Top 5 Most Important Features:")
for i in range(5):
    print(f"   {i+1}. {all_feats[top_idx[i]]:<35}  {importances[top_idx[i]]:.4f}")

## Step 13 · GridSearch Results Visualization

In [ ]:
# ============================================================
# CELL 14 ▸ GridSearch CV Results
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("GridSearchCV Results", fontsize=14, fontweight="bold")

# LR results
lr_results = pd.DataFrame(lr_gs.cv_results_)
lr_results['params_str'] = lr_results['params'].apply(
    lambda p: f"C={p['classifier__C']}\n{p['classifier__solver']}"
)
axes[0].bar(range(len(lr_results)), lr_results['mean_test_score'],
            yerr=lr_results['std_test_score'], color='#4A90D9',
            edgecolor='black', linewidth=0.5, capsize=4)
axes[0].set_xticks(range(len(lr_results)))
axes[0].set_xticklabels(lr_results['params_str'], fontsize=8, rotation=30, ha='right')
axes[0].set_title("Logistic Regression — CV F1 per Config", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Mean CV F1-Score"); axes[0].grid(True, alpha=0.3, axis='y')
axes[0].axhline(lr_gs.best_score_, color='red', linestyle='--', linewidth=1.5, label=f'Best: {lr_gs.best_score_:.4f}')
axes[0].legend()

# RF results — best few
rf_results = pd.DataFrame(rf_gs.cv_results_).sort_values('mean_test_score', ascending=False).head(6)
rf_results['params_str'] = rf_results['params'].apply(
    lambda p: f"n={p['classifier__n_estimators']}\nd={p['classifier__max_depth']}\nms={p['classifier__min_samples_split']}"
)
axes[1].bar(range(len(rf_results)), rf_results['mean_test_score'],
            yerr=rf_results['std_test_score'], color='#E74C3C',
            edgecolor='black', linewidth=0.5, capsize=4)
axes[1].set_xticks(range(len(rf_results)))
axes[1].set_xticklabels(rf_results['params_str'], fontsize=8, rotation=30, ha='right')
axes[1].set_title("Random Forest — Top 6 CV F1 Configs", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Mean CV F1-Score"); axes[1].grid(True, alpha=0.3, axis='y')
axes[1].axhline(rf_gs.best_score_, color='red', linestyle='--', linewidth=1.5, label=f'Best: {rf_gs.best_score_:.4f}')
axes[1].legend()

plt.tight_layout()
plt.savefig("gridsearch_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ GridSearch results plot saved!")

## Step 14 · Cross-Validation Score Comparison

In [ ]:
# ============================================================
# CELL 15 ▸ Cross-Validation Comparison
# ============================================================

cv_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("📊 5-Fold Cross-Validation on Full Training Set (Tuned Models):")
print("-" * 55)

cv_results = {}
for name, model in best_models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv_final,
                                scoring='f1', n_jobs=-1)
    cv_results[name] = cv_scores
    print(f"\n  {name}:")
    print(f"    Per-fold F1 : {[round(s,4) for s in cv_scores]}")
    print(f"    Mean F1     : {cv_scores.mean():.4f}")
    print(f"    Std F1      : {cv_scores.std():.4f} (±{cv_scores.std()*2:.4f})")

# Box plot of CV scores
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([cv_results[n] for n in best_models],
           labels=list(best_models.keys()),
           patch_artist=True,
           boxprops=dict(facecolor='#4A90D9', alpha=0.6),
           medianprops=dict(color='red', linewidth=2))
ax.set_title("5-Fold CV F1-Score Distribution", fontsize=13, fontweight="bold")
ax.set_ylabel("F1-Score"); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig("cv_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ CV comparison saved!")

## Step 15 · Export Pipelines with Joblib
Export both complete pipelines (preprocessing + model) as `.pkl` files.
These can be loaded in production with a **single line of code** — no re-training needed.

In [ ]:
# ============================================================
# CELL 16 ▸ Save Complete Pipelines with Joblib
# ============================================================

SAVE_DIR = "./saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save both pipelines
models_to_save = {
    "logistic_regression_pipeline": lr_gs.best_estimator_,
    "random_forest_pipeline":       rf_gs.best_estimator_,
}

for filename, pipeline in models_to_save.items():
    path = os.path.join(SAVE_DIR, f"{filename}.pkl")
    joblib.dump(pipeline, path)
    size = os.path.getsize(path) / 1e6
    print(f"✅ Saved: {path}  ({size:.2f} MB)")

print(f"\n📁 All files in {SAVE_DIR}/:")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f"   {f:<45} ({size:.2f} MB)")

## Step 16 · Reload & Production Inference
This simulates **real production usage** — load the pipeline from disk and predict on raw data.

In [ ]:
# ============================================================
# CELL 17 ▸ Reload Pipeline & Production Inference
# ============================================================

# Load the best model from disk
loaded_model = joblib.load(f"{SAVE_DIR}/logistic_regression_pipeline.pkl")
print("✅ Pipeline loaded from disk!")
print(f"   Type: {type(loaded_model)}")

# Simulate 5 new raw customer records (no preprocessing needed — pipeline handles it)
sample_customers = X_test.head(5).copy()
sample_labels    = y_test.head(5).values

# Predict
predictions  = loaded_model.predict(sample_customers)
probabilities = loaded_model.predict_proba(sample_customers)[:, 1]

print("\n" + "=" * 65)
print("🔮 PRODUCTION INFERENCE — 5 New Customer Records")
print("=" * 65)
print(f"{'#':<4} {'Actual':<12} {'Predicted':<12} {'Churn Prob':<12} {'Correct?'}")
print("-" * 55)
for i, (actual, pred, prob) in enumerate(zip(sample_labels, predictions, probabilities), 1):
    actual_lbl = "Churn" if actual == 1 else "No Churn"
    pred_lbl   = "Churn" if pred   == 1 else "No Churn"
    correct    = "✅" if actual == pred else "❌"
    print(f"{i:<4} {actual_lbl:<12} {pred_lbl:<12} {prob:.4f}       {correct}")

print("\n💡 This is production-ready:")
print("   loaded_model.predict(raw_dataframe)  ← one line, no manual preprocessing!")

## Step 17 · Final Model Comparison Table

In [ ]:
# ============================================================
# CELL 18 ▸ Final Comparison Summary
# ============================================================

comparison = []
for name, model in best_models.items():
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    comparison.append({
        "Model":            name,
        "Accuracy":         f"{accuracy_score(y_test, preds):.4f}",
        "F1-Score (Churn)": f"{f1_score(y_test, preds):.4f}",
        "ROC-AUC":          f"{roc_auc_score(y_test, proba):.4f}",
        "Best Params":      str({k.replace('classifier__',''):v for k,v in
                                 (lr_gs if 'Logistic' in name else rf_gs).best_params_.items()}),
    })

summary_df = pd.DataFrame(comparison).set_index("Model")
print("=" * 75)
print("📊 FINAL MODEL COMPARISON")
print("=" * 75)
print(summary_df.to_string())
print("=" * 75)

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
metrics   = ['Accuracy', 'F1-Score (Churn)', 'ROC-AUC']
colors    = ['#4A90D9','#E74C3C']

for ax, metric in zip(axes, metrics):
    vals  = [float(summary_df.loc[m, metric]) for m in summary_df.index]
    bars  = ax.bar(list(summary_df.index), vals, color=colors, edgecolor='black', linewidth=0.6, width=0.5)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=10)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle("Model Performance Comparison", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Comparison chart saved!")

---

## 📝 Project Summary & Insights

### ✅ What We Built
A **production-ready, end-to-end ML pipeline** for customer churn prediction using the Telco dataset.

---

### 📊 Results Summary

| Model | Accuracy | F1 (Churn) | ROC-AUC |
|-------|----------|------------|---------|
| Logistic Regression | ~80.6% | ~0.60 | ~0.842 |
| Random Forest | ~80.0% | ~0.58 | ~0.828 |

> Logistic Regression outperforms Random Forest on this dataset — a great reminder that simpler models can win!

---

### 🔍 Key Insights

1. **Tenure is the strongest predictor** — customers with short tenure churn most.
2. **Month-to-month contracts = high churn risk** — long-term contracts strongly retain customers.
3. **Fiber optic users churn more** — likely due to pricing, worth investigating.
4. **Class imbalance (26% churn)** — F1-Score is a better metric than accuracy for this.
5. **Pipeline = production-ready** — raw data in, prediction out, no manual preprocessing.

---

### 🏭 Pipeline Architecture

```
Raw CSV Data
     ↓
ColumnTransformer
  ├── Numeric: SimpleImputer(median) → StandardScaler
  └── Categorical: SimpleImputer(mode) → OneHotEncoder
     ↓
Classifier (LR / RF)
     ↓
Prediction + Probability
     ↓
joblib.dump() → .pkl file → Production
```

---

### 🚀 Possible Improvements
- Add **XGBoost / LightGBM** to the comparison
- Use **SMOTE** for class imbalance oversampling
- Try **RandomizedSearchCV** for faster tuning over larger grids
- Add **feature selection** step to the pipeline
- Deploy via **FastAPI / Flask** using the saved `.pkl` file

---

### 🛠️ Skills Demonstrated
✅ Scikit-learn Pipeline API  
✅ ColumnTransformer for mixed data types  
✅ Hyperparameter tuning with GridSearchCV  
✅ Stratified K-Fold cross-validation  
✅ Model evaluation (Accuracy, F1, AUC, Confusion Matrix, ROC)  
✅ Pipeline export & reload with joblib  
✅ Production inference simulation  

---
*DevelopersHub Corporation · AI/ML Engineering Internship · Task 2 · 2026*